# Pandas for Data Manipulation and Analysis - Part 2.2

### Credits
- Code and text by Antonio C. Briza (acbriza at gmail.com)<br>

- The outline of Lessons 6 and 7 of this workbook is mostly adapted from Chapter 5 of *Pandas for Data Analysis, 2nd Edition* by Wes McKinney.
- Other text and code come from [10 Minutes to pandas](http://pandas.pydata.org/pandas-docs/stable/10min.html), a tutorial from Panda's documentation website.  Code and text directly lifted from this source are indicated by `[TMP]`.


Last Modified: March 15, 2019<br>
[CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)

# Before you begin

First, make a copy of this notebook so that you can make changes as you please. Run and edit the copy instead of this original notebook. To copy this notebook, go to `File|Save a Copy to Drive`.

If this notebook is already a copy of the original, clear all outputs in this notebook. Go to the Menu and click on `Cell| All Outputs| Clear`. Once this is done, you're ready to go.

[あ]

<a id='contents'></a>

# TABLE OF CONTENTS

[05. Basic DataFrame Operations](#lesson5)<br>
[06. Indexing, Selection and Filtering](#lesson6)<br>
[07. Missing Data](#lesson7)<br>
[08. Operations](#lesson8)<br>

## Preliminaries

First, let's import pandas and numpy and set some display options and set plots to display in the notebook.

In [1]:
import numpy as np
import pandas as pd

np.set_printoptions(precision=4)   #set 4 decimal places for outputs from numpy
pd.options.display.max_rows=15      #display 6 rows for outputs from pandas

%matplotlib inline

<a id='lesson5'></a>

# 5. Basic DataFrame operations

Let's create a DataFrame for illustrating the operations in this lesson

### Dataset: Dummy Loan Data

In [2]:
dfL = pd.DataFrame({'amount':[5.2, 1.5, 0.75, 2.1, 3.2, 1.2],
                    'interest' :[0.14, 0.08, 0.06, 0.1, 0.12, 0.08],
                    'grade': ['B', 'A', 'C', 'A', 'D', 'A']},
                  index=['loan1','loan2','loan3','loan4','loan5','loan6'])
dfL

,amount,interest,grade
loan1,5.20,0.14,B
loan2,1.50,0.08,A
loan3,0.75,0.06,C
loan4,2.10,0.10,A
loan5,3.20,0.12,D
loan6,1.20,0.08,A


## 5.1 Retrieving Columns

Columns could be selected by using square brackets and specifying a column name (like the notation in a dictionary). This returns a `Series`.

In [3]:
dfL['amount']

,amount
loan1,5.20
loan2,1.50
loan3,0.75
loan4,2.10
loan5,3.20
loan6,1.20


In [4]:
type(dfL['amount'])

pandas.core.series.Series

In [5]:
dfL['grade']

,grade
loan1,B
loan2,A
loan3,C
loan4,A
loan5,D
loan6,A


Columns could also be accessed with the `dot` notation provided that the column name is a valid python identifier

In [6]:
dfL.amount

,amount
loan1,5.20
loan2,1.50
loan3,0.75
loan4,2.10
loan5,3.20
loan6,1.20


In [7]:
dfL.grade

,grade
loan1,B
loan2,A
loan3,C
loan4,A
loan5,D
loan6,A


## 5.2 Retrieving Rows

Rows can be retrieved via their **labels** using the  **`loc`** indexer.  This will be discussed in greater detail later in this notebook.

In [8]:
dfL.loc['loan1']

,loan1
amount,5.2
interest,0.14
grade,B


In [9]:
dfL.loc['loan4']

,loan4
amount,2.1
interest,0.1
grade,A


## 5.3 Modifying Columns with an  Assignment Expression

Columns could be modified by assignment

### Column Assignment with a scalar

All rows are assigned the scalar value

In [10]:
dfL['grade'] = 'C'
dfL

,amount,interest,grade
loan1,5.20,0.14,C
loan2,1.50,0.08,C
loan3,0.75,0.06,C
loan4,2.10,0.10,C
loan5,3.20,0.12,C
loan6,1.20,0.08,C


### Column Assignment with a list

Assigning with `lists` or numpy.ndarray requires that the length (the number rows) of the DataFrame is matched

In [11]:
v = np.random.rand(6)
dfL['interest'] = v
dfL

,amount,interest,grade
loan1,5.20,0.837861,C
loan2,1.50,0.070973,C
loan3,0.75,0.456866,C
loan4,2.10,0.686649,C
loan5,3.20,0.275397,C
loan6,1.20,0.335527,C


In [12]:
# This code will produce an error since the dimensions don't match
try:
    dfL['interest'] = [0.13, 0.07, 0.05, 0.09, 0.11]
except ValueError as error:
    print('Assignment error: ', error)

Assignment error:  Length of values (5) does not match length of index (6)


The list below has the matching dimensions

In [13]:
dfL['interest'] = [0.13, 0.07, 0.05, 0.09, 0.11, 0.07]
dfL

,amount,interest,grade
loan1,5.20,0.13,C
loan2,1.50,0.07,C
loan3,0.75,0.05,C
loan4,2.10,0.09,C
loan5,3.20,0.11,C
loan6,1.20,0.07,C


### Column Assignment with a Series

Assignment with a Series matches the index. Any unmatched index will be **"plugged"** with `np.nan` values

In [14]:
s = pd.Series([0.15, 0.09, 0.11,  0.09],
              index=['loan1', 'loan2','loan4', 'loan6'])
dfL['interest'] = s
dfL

,amount,interest,grade
loan1,5.20,0.15,C
loan2,1.50,0.09,C
loan3,0.75,NaN,C
loan4,2.10,0.11,C
loan5,3.20,NaN,C
loan6,1.20,0.09,C


## 5.4 Adding a New Column with an  Assignment Expression

An assignment expression for a **column that does not exist** adds a new column to the DataFrame.

In [15]:
dfL['purpose'] = ['House','Car','Medical','Car','House', 'Medical']
dfL

,amount,interest,grade,purpose
loan1,5.20,0.15,C,House
loan2,1.50,0.09,C,Car
loan3,0.75,NaN,C,Medical
loan4,2.10,0.11,C,Car
loan5,3.20,NaN,C,House
loan6,1.20,0.09,C,Medical


## 5.5 Adding a New Row with `Concat`

A new row could be added to a DataFrame. To do this, pass a `Series` with a `name` to `pd.concat()`

In [24]:
help(pd.concat)

Help on function concat in module pandas.core.reshape.concat:

concat(
    objs: 'Iterable[Series | DataFrame] | Mapping[HashableT, Series | DataFrame]',
    *,
    axis: 'Axis' = 0,
    join: 'str' = 'outer',
    ignore_index: 'bool' = False,
    keys: 'Iterable[Hashable] | None' = None,
    levels=None,
    names: 'list[HashableT] | None' = None,
    verify_integrity: 'bool' = False,
    sort: 'bool' = False,
    copy: 'bool | None' = None
) -> 'DataFrame | Series'
    Concatenate pandas objects along a particular axis.

    Allows optional set logic along the other axes.

    Can also add a layer of hierarchical indexing on the concatenation axis,
    which may be useful if the labels are the same (or overlapping) on
    the passed axis number.

    Parameters
    ----------
    objs : a sequence or mapping of Series or DataFrame objects
        If a mapping is passed, the sorted keys will be used as the `keys`
        argument, unless it is passed, in which case the values will be


In [17]:
new_loan = pd.Series({'amount' : 0.6,
                      'interest' : 0.12,
                      'grade' : 'B',
                      'purpose' : 'Vacation'},
                     name='loan7')

In [18]:
new_loan

,loan7
amount,0.6
interest,0.12
grade,B
purpose,Vacation


In [26]:
# Adding a row using concat
dfL = pd.concat([dfL,new_loan.to_frame().T])
dfL

,amount,interest,grade,purpose
loan1,5.2,0.15,C,House
loan2,1.5,0.09,C,Car
loan3,0.75,NaN,C,Medical
loan4,2.1,0.11,C,Car
loan5,3.2,NaN,C,House
loan6,1.2,0.09,C,Medical
loan7,0.6,0.12,B,Vacation


## 5.6 Deleting Rows and Columns

One or more rows could be deleted using the `DataFrame.drop()` method. The default axis is `rows` (this will drop `rows`).  Passing `axis=columns` (or `axis=1`)  will drop columns.

In [27]:
help(dfL.drop)

Help on method drop in module pandas.core.frame:

drop(
    labels: 'IndexLabel | None' = None,
    *,
    axis: 'Axis' = 0,
    index: 'IndexLabel | None' = None,
    columns: 'IndexLabel | None' = None,
    level: 'Level | None' = None,
    inplace: 'bool' = False,
    errors: 'IgnoreRaise' = 'raise'
) -> 'DataFrame | None' method of pandas.core.frame.DataFrame instance
    Drop specified labels from rows or columns.

    Remove rows or columns by specifying label names and corresponding
    axis, or by directly specifying index or column names. When using a
    multi-index, labels on different levels can be removed by specifying
    the level. See the :ref:`user guide <advanced.shown_levels>`
    for more information about the now unused levels.

    Parameters
    ----------
    labels : single label or list-like
        Index or column labels to drop. A tuple will be used as a single
        label and not treated as a list-like.
    axis : {0 or 'index', 1 or 'columns'}, default 0

### Dropping Rows

First let's copy our DataFrame

In [33]:
dfL_temp = dfL.copy()
dfL_temp

,amount,interest,grade,purpose
loan1,5.2,0.15,C,House
loan2,1.5,0.09,C,Car
loan3,0.75,NaN,C,Medical
loan4,2.1,0.11,C,Car
loan5,3.2,NaN,C,House
loan6,1.2,0.09,C,Medical
loan7,0.6,0.12,B,Vacation


In [34]:
dfL_temp.drop('loan6')
dfL_temp

,amount,interest,grade,purpose
loan1,5.2,0.15,C,House
loan2,1.5,0.09,C,Car
loan3,0.75,NaN,C,Medical
loan4,2.1,0.11,C,Car
loan5,3.2,NaN,C,House
loan6,1.2,0.09,C,Medical
loan7,0.6,0.12,B,Vacation


Q: Why is row `loan6` still there?

A: The `drop()` method returns the rows/columns that are dropped. Since we did assign it to any variable, the results were lost.

Now lets do the same, but re-assign the result to `dfL_temp`

In [35]:
dfL_temp = dfL_temp.drop('loan6')
dfL_temp

,amount,interest,grade,purpose
loan1,5.2,0.15,C,House
loan2,1.5,0.09,C,Car
loan3,0.75,NaN,C,Medical
loan4,2.1,0.11,C,Car
loan5,3.2,NaN,C,House
loan7,0.6,0.12,B,Vacation


Another way of doing this is by passing the parameter `inplace=True`. Let's try it by deleting `loan5`

In [36]:
dfL_temp.drop('loan5', inplace=True)
dfL_temp

,amount,interest,grade,purpose
loan1,5.2,0.15,C,House
loan2,1.5,0.09,C,Car
loan3,0.75,NaN,C,Medical
loan4,2.1,0.11,C,Car
loan7,0.6,0.12,B,Vacation


You can drop multiple rows by passing a list of labels

In [ ]:
dfL_temp.drop(['loan2', 'loan3'], inplace=True)
dfL_temp

### Dropping Columns

Columns could be passed by passing `axis=columns` or `axis=1` to `drop()`

In [37]:
dfL_temp2 = dfL.copy()
dfL_temp2

,amount,interest,grade,purpose
loan1,5.2,0.15,C,House
loan2,1.5,0.09,C,Car
loan3,0.75,NaN,C,Medical
loan4,2.1,0.11,C,Car
loan5,3.2,NaN,C,House
loan6,1.2,0.09,C,Medical
loan7,0.6,0.12,B,Vacation


In [38]:
dfL_temp2.drop(['purpose', 'grade'], axis='columns', inplace=True)
dfL_temp2

,amount,interest
loan1,5.2,0.15
loan2,1.5,0.09
loan3,0.75,NaN
loan4,2.1,0.11
loan5,3.2,NaN
loan6,1.2,0.09
loan7,0.6,0.12


[TABLE OF CONTENTS](#contents)

<a id='lesson6'></a>

# 6. Indexing, Selection and Filtering

## 6.1 Reindexing, Setting, and Resetting Indices

### reindex()

In [39]:
help(dfL.reindex)

Help on method reindex in module pandas.core.frame:

reindex(
    labels=None,
    *,
    index=None,
    columns=None,
    axis: 'Axis | None' = None,
    method: 'ReindexMethod | None' = None,
    copy: 'bool | None' = None,
    level: 'Level | None' = None,
    fill_value: 'Scalar | None' = nan,
    limit: 'int | None' = None,
    tolerance=None
) -> 'DataFrame' method of pandas.core.frame.DataFrame instance
    Conform DataFrame to new index with optional filling logic.

    Places NA/NaN in locations having no value in the previous index. A new object
    is produced unless the new index is equivalent to the current one and
    ``copy=False``.

    Parameters
    ----------

    labels : array-like, optional
        New labels / index to conform the axis specified by 'axis' to.
    index : array-like, optional
        New labels for the index. Preferably an Index object to avoid
        duplicating data.
    columns : array-like, optional
        New labels for the columns. Prefer

 > Conform DataFrame to new index with optional filling logic, placing
    NA/NaN in locations having no value in the previous index. A **new object
    is produced** ...

#### Reindexing rows

Let's get a new object with rows `loan1` and `loan2` removed, then add `loan8` and `loan9`

In [41]:
obj1 = dfL.reindex(['loan3','loan4','loan5','loan6','loan7','loan8', 'loan9'])
obj1

,amount,interest,grade,purpose
loan3,0.75,NaN,C,Medical
loan4,2.1,0.11,C,Car
loan5,3.2,NaN,C,House
loan6,1.2,0.09,C,Medical
loan7,0.6,0.12,B,Vacation
loan8,NaN,NaN,NaN,NaN
loan9,NaN,NaN,NaN,NaN


#### Reindexing columns

Let's get a new object with `grade` column removed, then add a column for `occupation`

In [42]:
obj2 = dfL.reindex(columns=['amount','interest','purpose','occupation'])
obj2

,amount,interest,purpose,occupation
loan1,5.2,0.15,House,NaN
loan2,1.5,0.09,Car,NaN
loan3,0.75,NaN,Medical,NaN
loan4,2.1,0.11,Car,NaN
loan5,3.2,NaN,House,NaN
loan6,1.2,0.09,Medical,NaN
loan7,0.6,0.12,Vacation,NaN


### set_index()

In [43]:
help(dfL.set_index)

Help on method set_index in module pandas.core.frame:

set_index(
    keys,
    *,
    drop: 'bool' = True,
    append: 'bool' = False,
    inplace: 'bool' = False,
    verify_integrity: 'bool' = False
) -> 'DataFrame | None' method of pandas.core.frame.DataFrame instance
    Set the DataFrame index using existing columns.

    Set the DataFrame index (row labels) using one or more existing
    columns or arrays (of the correct length). The index can replace the
    existing index or expand on it.

    Parameters
    ----------
    keys : label or array-like or list of labels/arrays
        This parameter can be either a single column key, a single array of
        the same length as the calling DataFrame, or a list containing an
        arbitrary combination of column keys and arrays. Here, "array"
        encompasses :class:`Series`, :class:`Index`, ``np.ndarray``, and
        instances of :class:`~collections.abc.Iterator`.
    drop : bool, default True
        Delete columns to be 

 > Set the DataFrame index (row labels) using one or more existing columns. By default **yields a new object**.

In [44]:
dfL

,amount,interest,grade,purpose
loan1,5.2,0.15,C,House
loan2,1.5,0.09,C,Car
loan3,0.75,NaN,C,Medical
loan4,2.1,0.11,C,Car
loan5,3.2,NaN,C,House
loan6,1.2,0.09,C,Medical
loan7,0.6,0.12,B,Vacation


In [45]:
obj3 = dfL.set_index('purpose')
obj3

,amount,interest,grade
purpose,,,
House,5.2,0.15,C
Car,1.5,0.09,C
Medical,0.75,NaN,C
Car,2.1,0.11,C
House,3.2,NaN,C
Medical,1.2,0.09,C
Vacation,0.6,0.12,B


As you can see, `pandas` allow duplicate indices.

Notice also that running the code below will not alter `dfL` (in fact it will do nothing as the return value of the set_index is not captured in a new variable)

In [46]:
dfL.set_index('purpose')
dfL

,amount,interest,grade,purpose
loan1,5.2,0.15,C,House
loan2,1.5,0.09,C,Car
loan3,0.75,NaN,C,Medical
loan4,2.1,0.11,C,Car
loan5,3.2,NaN,C,House
loan6,1.2,0.09,C,Medical
loan7,0.6,0.12,B,Vacation


### reset_index()

Using `reset_index()` on a DataFrame with 1-level of index adds the old index as a column, then a adds a new numeric index. It returns a new DataFrame.

In [47]:
help(dfL.reset_index)

Help on method reset_index in module pandas.core.frame:

reset_index(
    level: 'IndexLabel | None' = None,
    *,
    drop: 'bool' = False,
    inplace: 'bool' = False,
    col_level: 'Hashable' = 0,
    col_fill: 'Hashable' = '',
    allow_duplicates: 'bool | lib.NoDefault' = <no_default>,
    names: 'Hashable | Sequence[Hashable] | None' = None
) -> 'DataFrame | None' method of pandas.core.frame.DataFrame instance
    Reset the index, or a level of it.

    Reset the index of the DataFrame, and use the default one instead.
    If the DataFrame has a MultiIndex, this method can remove one or more
    levels.

    Parameters
    ----------
    level : int, str, tuple, or list, default None
        Only remove the given levels from the index. Removes all levels by
        default.
    drop : bool, default False
        Do not try to insert index into dataframe columns. This resets
        the index to the default integer index.
    inplace : bool, default False
        Whether to modi

In [48]:
obj3 = dfL.reset_index()
obj3

,index,amount,interest,grade,purpose
0,loan1,5.2,0.15,C,House
1,loan2,1.5,0.09,C,Car
2,loan3,0.75,NaN,C,Medical
3,loan4,2.1,0.11,C,Car
4,loan5,3.2,NaN,C,House
5,loan6,1.2,0.09,C,Medical
6,loan7,0.6,0.12,B,Vacation


## 6.2 Series Indexing and Slicing

Let's get a Series by getting a view of one of the columns of our loan DataFrame

In [49]:
s = dfL['amount']
s

,amount
loan1,5.2
loan2,1.5
loan3,0.75
loan4,2.1
loan5,3.2
loan6,1.2
loan7,0.6


### Indexing and Slicing with **Numeric Indices**

Indexing and Slicing a series with **numerical indices** works like NumPy array. However, to avoid ambiguity and future warnings, use `.iloc[]` for positional indexing.

In [58]:
s.iloc[0]

5.2

In [59]:
s.iloc[-1]

0.6

In [60]:
s.iloc[1:4]

,amount
loan2,1.5
loan3,0.75
loan4,2.1


Indexing using a list of integer indices

In [61]:
s.iloc[[1,3,5]]

,amount
loan2,1.5
loan4,2.1
loan6,1.2


We can also index using a boolean indexing/masking

In [65]:
s > 1.5

,amount
loan1,True
loan2,False
loan3,False
loan4,True
loan5,True
loan6,False
loan7,False


In [64]:
#indexing using a boolean index/mask
s[s > 1.5]

,amount
loan1,5.2
loan4,2.1
loan5,3.2


#### Indexing and Slicing with **row labels**

However, for pandas Series, you could also index and slice using row labels

In [68]:
s['loan1']

5.2

In [69]:
s['loan6']

1.2

Slicing a series with row labels differs from NumPy in that **endpoints are included**

In [70]:
s['loan2':'loan5']

,amount
loan2,1.5
loan3,0.75
loan4,2.1
loan5,3.2


Indexing using a list of labels

In [71]:
s[['loan2', 'loan4', 'loan6']]

,amount
loan2,1.5
loan4,2.1
loan6,1.2


## 6.3 DataFrame Indexing

Dataframe indexing works on columns. It returns a Series or a DataFrame.

In [72]:
dfL

,amount,interest,grade,purpose
loan1,5.2,0.15,C,House
loan2,1.5,0.09,C,Car
loan3,0.75,NaN,C,Medical
loan4,2.1,0.11,C,Car
loan5,3.2,NaN,C,House
loan6,1.2,0.09,C,Medical
loan7,0.6,0.12,B,Vacation


In [73]:
dfL['amount']

,amount
loan1,5.2
loan2,1.5
loan3,0.75
loan4,2.1
loan5,3.2
loan6,1.2
loan7,0.6


In [74]:
dfL[['amount','interest']]

,amount,interest
loan1,5.2,0.15
loan2,1.5,0.09
loan3,0.75,NaN
loan4,2.1,0.11
loan5,3.2,NaN
loan6,1.2,0.09
loan7,0.6,0.12


**You cannot select rows by indexing a DataFrame**. For this, you have to use the `loc` or `iloc` method (more on these below). Thus, the following will generate an error:

In [76]:
try:
    dfL['loan1']
except KeyError as error:
    print('Key Error indexing a DataFrame:', error)

Key Error indexing a DataFrame: 'loan1'


However, you can generate **row slices** with **numeric indexing** on DataFrames:

In [77]:
dfL[:3]

,amount,interest,grade,purpose
loan1,5.2,0.15,C,House
loan2,1.5,0.09,C,Car
loan3,0.75,NaN,C,Medical


In [78]:
dfL[::2]

,amount,interest,grade,purpose
loan1,5.2,0.15,C,House
loan3,0.75,NaN,C,Medical
loan5,3.2,NaN,C,House
loan7,0.6,0.12,B,Vacation


Rows could also be selected using boolean indexing/masking

In [79]:
mask = dfL['interest'] > .1
mask

,interest
loan1,True
loan2,False
loan3,False
loan4,True
loan5,False
loan6,False
loan7,True


In [80]:
dfL[mask]

,amount,interest,grade,purpose
loan1,5.2,0.15,C,House
loan4,2.1,0.11,C,Car
loan7,0.6,0.12,B,Vacation


In [81]:
dfL[dfL['interest']>0.1]

,amount,interest,grade,purpose
loan1,5.2,0.15,C,House
loan4,2.1,0.11,C,Car
loan7,0.6,0.12,B,Vacation


## 6.4 Numpy-like Slicing with `loc` and `iloc`

#### Review

As a quick review, remember the following:

**Series**
- integer indexing and slicing work like NumPy arrays
- label indexing is also possible
- label slicing include the endpoints, unlike index-slicing in NumPy arrays
- slicing using a list of indices or labels
- slicing using a boolean indexing is possible

**DataFrame**
- integer indexing is **not possible**. (For this you have to use `iloc[]`)
- label indexing selects **columns**
- slicing using a list of column labels is possible
    - this selects columns
- slicing using integer slices is possible
    - this selects rows
    - this is made avaliable for convenience
    - the more general method is to use `iloc[]`
- slicing using a boolean indexing is possible
    - this selects rows

#### Introducing `loc[]` and `iloc[]` indexing operators

The `loc[]` and `iloc[]` indexing operators exist to  provide NumPy-like notation for slicing DataFrames by using row/column labels (via `loc`) or integer positions (via `iloc`).

### 6.4.1 Label Slicing with `loc[]`

In [82]:
dfL

,amount,interest,grade,purpose
loan1,5.2,0.15,C,House
loan2,1.5,0.09,C,Car
loan3,0.75,NaN,C,Medical
loan4,2.1,0.11,C,Car
loan5,3.2,NaN,C,House
loan6,1.2,0.09,C,Medical
loan7,0.6,0.12,B,Vacation


#### Slices of Rows

In [83]:
dfL.loc[:'loan3'] # dfL.loc[:'loan3',] or dfL.loc[:'loan3',:] will give the same output

,amount,interest,grade,purpose
loan1,5.2,0.15,C,House
loan2,1.5,0.09,C,Car
loan3,0.75,NaN,C,Medical


In [84]:
dfL.loc['loan2':'loan4'] #notice that the endpoint IS included

,amount,interest,grade,purpose
loan2,1.5,0.09,C,Car
loan3,0.75,NaN,C,Medical
loan4,2.1,0.11,C,Car


In [85]:
dfL.loc['loan1':'loan5':2] #notice that the endpoint IS included

,amount,interest,grade,purpose
loan1,5.2,0.15,C,House
loan3,0.75,NaN,C,Medical
loan5,3.2,NaN,C,House


In [86]:
dfL.loc[['loan1','loan5']]

,amount,interest,grade,purpose
loan1,5.2,0.15,C,House
loan5,3.2,NaN,C,House


#### Slices of Columns

To specify column slices, the first element of the slice has to contain `:,` which selects all rows in the DataFrame.

In [87]:
dfL.loc[:, :'interest']

,amount,interest
loan1,5.2,0.15
loan2,1.5,0.09
loan3,0.75,NaN
loan4,2.1,0.11
loan5,3.2,NaN
loan6,1.2,0.09
loan7,0.6,0.12


Omitting `:,` above will produce a DataFrame without rows. This is because you only provided the row slice, and since none of the row labels matches, you get an empty DataFrame.

In [88]:
dfL.loc[:'interest']

,amount,interest,grade,purpose


We fix it by adding `:,`, as mentioned earlier.

In [89]:
dfL.loc[:, 'interest':]

,interest,grade,purpose
loan1,0.15,C,House
loan2,0.09,C,Car
loan3,NaN,C,Medical
loan4,0.11,C,Car
loan5,NaN,C,House
loan6,0.09,C,Medical
loan7,0.12,B,Vacation


In [90]:
dfL.loc[:, ['amount', 'interest']]

,amount,interest
loan1,5.2,0.15
loan2,1.5,0.09
loan3,0.75,NaN
loan4,2.1,0.11
loan5,3.2,NaN
loan6,1.2,0.09
loan7,0.6,0.12


#### "block" slices

Combining row and column slicing can help us create 'block' slices

In [91]:
dfL.loc['loan4':, :'interest']

,amount,interest
loan4,2.1,0.11
loan5,3.2,NaN
loan6,1.2,0.09
loan7,0.6,0.12


In [92]:
dfL.loc[['loan1','loan3'], ['amount','interest']]

,amount,interest
loan1,5.2,0.15
loan3,0.75,NaN


### 6.4.2 Positional Slicing with `iloc[]`

`iloc` works like `loc`, but for integer indices instead of labels. Keep in mind that endpoints are not included in `iloc` indexing.

In [93]:
dfL

,amount,interest,grade,purpose
loan1,5.2,0.15,C,House
loan2,1.5,0.09,C,Car
loan3,0.75,NaN,C,Medical
loan4,2.1,0.11,C,Car
loan5,3.2,NaN,C,House
loan6,1.2,0.09,C,Medical
loan7,0.6,0.12,B,Vacation


#### Slices of rows

In [94]:
dfL.iloc[:3] # dfL.iloc[:3,] or dfL.iloc[:3,:] will give the same output

,amount,interest,grade,purpose
loan1,5.2,0.15,C,House
loan2,1.5,0.09,C,Car
loan3,0.75,NaN,C,Medical


Notice that `index 3` (row label `loan4`) is not included in the slice

In [95]:
dfL.iloc[1:3] #notice that the endpoint is NOT included

,amount,interest,grade,purpose
loan2,1.5,0.09,C,Car
loan3,0.75,NaN,C,Medical


In [96]:
dfL.iloc[:4:2] #notice that the endpoint is NOT included

,amount,interest,grade,purpose
loan1,5.2,0.15,C,House
loan3,0.75,NaN,C,Medical


In [97]:
dfL.iloc[[0,4]]

,amount,interest,grade,purpose
loan1,5.2,0.15,C,House
loan5,3.2,NaN,C,House


#### slices of columns

To specify column slices, the first element of the slice has to contain `:,` which will select all rows.

In [98]:
dfL.iloc[:, :2]

,amount,interest
loan1,5.2,0.15
loan2,1.5,0.09
loan3,0.75,NaN
loan4,2.1,0.11
loan5,3.2,NaN
loan6,1.2,0.09
loan7,0.6,0.12


Omitting `:,` above will select rows. This is because you are actually providing a row slice.

In [99]:
dfL.iloc[:2]

,amount,interest,grade,purpose
loan1,5.2,0.15,C,House
loan2,1.5,0.09,C,Car


In [100]:
dfL.iloc[:, 2:]

,grade,purpose
loan1,C,House
loan2,C,Car
loan3,C,Medical
loan4,C,Car
loan5,C,House
loan6,C,Medical
loan7,B,Vacation


In [101]:
dfL.iloc[:, [0, 1]]

,amount,interest
loan1,5.2,0.15
loan2,1.5,0.09
loan3,0.75,NaN
loan4,2.1,0.11
loan5,3.2,NaN
loan6,1.2,0.09
loan7,0.6,0.12


#### "block" slices

Combining row and column slicing can help us create 'block' slices

In [102]:
dfL.iloc[3:, :2]

,amount,interest
loan4,2.1,0.11
loan5,3.2,NaN
loan6,1.2,0.09
loan7,0.6,0.12


In [103]:
dfL.iloc[[0,4], [0,1]]

,amount,interest
loan1,5.2,0.15
loan5,3.2,NaN


## 6.5 Selecting Single Values with `at` and `iat`

### with `at` and `iat`

While `loc` and `iloc` can be used to select single values...

In [104]:
dfL.loc['loan1', 'amount']

5.2

In [105]:
dfL.iloc[0, 0]

5.2

for faster accessing of a single value (e.g. for making production-level code), it is better to use `at` and `iat`:

In [106]:
dfL.at['loan1', 'amount']

5.2

In [ ]:
dfL.iat[0, 0]

Compare time for executing  for 10000 times

In [107]:
%%timeit -n 10000
dfL.loc['loan1', 'amount']

5.33 µs ± 154 ns per loop (mean ± std. dev. of 7 runs, 10000 loops each)


In [108]:
%%timeit -n 10000
dfL.at['loan1', 'amount']

3.93 µs ± 927 ns per loop (mean ± std. dev. of 7 runs, 10000 loops each)


[TABLE OF CONTENTS](#contents)

<a id='lesson7'></a>

# 7. Missing Data

Let's take a look at another dataset

In [109]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [110]:
%cd /content/drive/My\ Drive/ColabNotebooks/ICT110-AppDev/ColabNotes

/content/drive/My Drive/ColabNotebooks/ICT110-AppDev/ColabNotes


In [137]:
ls Datasets/

 Energy_Indicators.xls           Loan_Default.csv
 EnergyXLS.jpg                   temp_data.csv
 gdp_all_na_dropped.xlsx         temp_data.xlsx
'GDP(PPP)_LifeExpectancy.xlsx'   WorldBank_WHO_Country_Dataset_2015.csv
 hprice1_ex8-4.csv


In [138]:
dfG = pd.read_csv('Datasets/WorldBank_WHO_Country_Dataset_2015.csv',
                  usecols =[0,1,2,3,4],    # use column indices 1,2,3,4,5
                  names=[
                      'Country',
                      'Life_Exp',
                      'GDP',
                      'Population',
                      'Inc_Class'
                      ],                  #column names
                  na_values=['', '.'],   # invalid (N/A) values
                  header=0
)
dfG

,Country,Life_Exp,GDP,Population,Inc_Class
0,ABW,75.405000,2.962907e+09,107906,H
1,AFE,61.713031,9.099053e+11,607123269,NaN
2,AFG,62.270000,1.913422e+10,33831764,L
3,AFW,56.033996,7.780375e+11,417536152,NaN
4,AGO,61.042000,1.025431e+11,28157798,LM
...,...,...,...,...,...
256,XKX,76.024000,6.295845e+09,1788274,UM
257,YEM,66.761000,4.244449e+10,31159379,L
258,ZAF,64.053000,3.467098e+11,56723537,UM
259,ZMB,60.728000,2.125122e+10,16399089,LM


In [144]:
dfG.shape

(261, 5)

In [145]:
dfG.describe()

,Life_Exp,GDP,Population
count,261.000000,2.560000e+02,2.610000e+02
mean,71.771971,2.131059e+12,2.779660e+08
std,8.016338,7.624593e+12,9.180351e+08
min,39.757000,3.675529e+07,1.095400e+04
25%,65.909000,8.665418e+09,1.453923e+06
50%,72.932000,4.806571e+10,9.797755e+06
75%,77.451220,4.657971e+11,5.108906e+07
max,85.323000,7.583935e+13,7.441085e+09


#### Energy_Indicators.xls

In [141]:
dEG = pd.read_excel(
    'Datasets/Energy_Indicators.xls',
    skiprows=17,
    skipfooter=38,
    usecols=list(range(2, 6)),
    names=[
        'Country',
        'Energy Supply',
        'Energy Supply per Capita',
        '% Renewable'
    ],
    na_values='...')
dEG

,Country,Energy Supply,Energy Supply per Capita,% Renewable
0,Afghanistan,321.0,10.0,78.669280
1,Albania,102.0,35.0,100.000000
2,Algeria,1959.0,51.0,0.551010
3,American Samoa,NaN,NaN,0.641026
4,Andorra,9.0,121.0,88.695650
...,...,...,...,...
222,Viet Nam,2554.0,28.0,45.321520
223,Wallis and Futuna Islands,0.0,26.0,0.000000
224,Yemen,344.0,13.0,0.000000
225,Zambia,400.0,26.0,99.714670


In [146]:
dEG.shape

(227, 4)

In [147]:
dEG.describe()

,Energy Supply,Energy Supply per Capita,% Renewable
count,222.000000,222.000000,227.000000
mean,2457.981982,90.666667,28.086077
std,11039.145412,116.234887,31.903505
min,0.000000,2.000000,0.000000
25%,33.000000,21.250000,0.022893
50%,189.500000,51.000000,14.814810
75%,996.250000,117.750000,50.168620
max,127191.000000,957.000000,100.000000


## 7.1 Dropping Missing Data

You can drop columns or rows using `dropna()` method

In [117]:
help(dfG.dropna)

Help on method dropna in module pandas.core.frame:

dropna(
    *,
    axis: 'Axis' = 0,
    how: 'AnyAll | lib.NoDefault' = <no_default>,
    thresh: 'int | lib.NoDefault' = <no_default>,
    subset: 'IndexLabel | None' = None,
    inplace: 'bool' = False,
    ignore_index: 'bool' = False
) -> 'DataFrame | None' method of pandas.core.frame.DataFrame instance
    Remove missing values.

    See the :ref:`User Guide <missing_data>` for more on which values are
    considered missing, and how to work with missing data.

    Parameters
    ----------
    axis : {0 or 'index', 1 or 'columns'}, default 0
        Determine if rows or columns which contain missing values are
        removed.

        * 0, or 'index' : Drop rows which contain missing values.
        * 1, or 'columns' : Drop columns which contain missing value.

        Only a single axis is allowed.

    how : {'any', 'all'}, default 'any'
        Determine if row or column is removed from DataFrame, when we have
        at le

### 7.1.1 Dropping all rows with invalid values

Let's drop all rows with invalid values and save the result in a file for manual verification.

In [148]:
df_temp = dfG.dropna()
df_temp.to_excel('Datasets/gdp_all_na_dropped.xlsx')
df_temp.describe()

,Life_Exp,GDP,Population
count,212.000000,2.120000e+02,2.120000e+02
mean,72.033189,3.551111e+11,3.485301e+07
std,8.253903,1.557175e+12,1.363125e+08
min,39.757000,3.675529e+07,1.095400e+04
25%,65.975000,6.174765e+09,8.786288e+05
50%,73.539195,2.337504e+10,6.344120e+06
75%,78.158000,1.695519e+11,2.302656e+07
max,85.323000,1.829502e+13,1.379860e+09


Notice that `count` has decreased in the  summary statistics above. We can verify this by checking the Excel file.

### 7.1.2 Dropping all rows with invalid values for a specific column

Let's drop only the rows with invalid values for the `GDP` column

In [149]:
dfG

,Country,Life_Exp,GDP,Population,Inc_Class
0,ABW,75.405000,2.962907e+09,107906,H
1,AFE,61.713031,9.099053e+11,607123269,NaN
2,AFG,62.270000,1.913422e+10,33831764,L
3,AFW,56.033996,7.780375e+11,417536152,NaN
4,AGO,61.042000,1.025431e+11,28157798,LM
...,...,...,...,...,...
256,XKX,76.024000,6.295845e+09,1788274,UM
257,YEM,66.761000,4.244449e+10,31159379,L
258,ZAF,64.053000,3.467098e+11,56723537,UM
259,ZMB,60.728000,2.125122e+10,16399089,LM


In [150]:
df_temp = dfG.dropna(subset=['GDP'])
df_temp.to_excel('gdp_na_dropped.xlsx')
df_temp.describe()

,Life_Exp,GDP,Population
count,256.000000,2.560000e+02,2.560000e+02
mean,71.698029,2.131059e+12,2.832825e+08
std,8.035365,7.624593e+12,9.261914e+08
min,39.757000,3.675529e+07,1.095400e+04
25%,65.783500,8.665418e+09,1.787820e+06
50%,72.915500,4.806571e+10,1.007863e+07
75%,77.382805,4.657971e+11,5.319661e+07
max,85.323000,7.583935e+13,7.441085e+09


### 7.1.3 Dropping after a `where` operation

In [152]:
help(dfG.where)

Help on method where in module pandas.core.generic:

where(
    cond,
    other=nan,
    *,
    inplace: 'bool_t' = False,
    axis: 'Axis | None' = None,
    level: 'Level | None' = None
) -> 'Self | None' method of pandas.core.frame.DataFrame instance
    Replace values where the condition is False.

    Parameters
    ----------
    cond : bool Series/DataFrame, array-like, or callable
        Where `cond` is True, keep the original value. Where
        False, replace with corresponding value from `other`.
        If `cond` is callable, it is computed on the Series/DataFrame and
        should return boolean Series/DataFrame or array. The callable must
        not change input Series/DataFrame (though pandas doesn't check it).
    other : scalar, Series/DataFrame, or callable
        Entries where `cond` is False are replaced with
        corresponding value from `other`.
        If other is callable, it is computed on the Series/DataFrame and
        should return scalar or Series/

`DataFrames.where` allows you, by default, to set a condition such that
- rows meeting the condition are not touched
- rows not meeting the condition are set to `NaN` (vs. being dropped in Boolean Indexing)

In [153]:
dfG.where(dfG.Life_Exp > 70)

,Country,Life_Exp,GDP,Population,Inc_Class
0,ABW,75.405,2.962907e+09,107906.0,H
1,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...
256,XKX,76.024,6.295845e+09,1788274.0,UM
257,NaN,NaN,NaN,NaN,NaN
258,NaN,NaN,NaN,NaN,NaN
259,NaN,NaN,NaN,NaN,NaN


Combined with `dropna()`, `where()` is a useful tool that can be used for **method chaining** in reformating a DataFrame

In [154]:
dfG_high_LE = dfG.where(dfG.Life_Exp>83).dropna()
dfG_high_LE

,Country,Life_Exp,GDP,Population,Inc_Class
6,AND,84.532000,2.789881e+09,72174.0,H
95,HKG,84.252683,3.093856e+11,7291300.0,H
117,JPN,83.793902,4.534439e+12,127141000.0,H
142,MAC,83.224390,4.494061e+10,642900.0,H
145,MCO,85.323000,6.264754e+09,36757.0,H
208,SMR,84.596000,1.419400e+09,32897.0,H


## 7.2 Filling Missing Data

Data could be filled with `fillna()`

In [155]:
help(dfG.fillna)

Help on method fillna in module pandas.core.generic:

fillna(
    value: 'Hashable | Mapping | Series | DataFrame | None' = None,
    *,
    method: 'FillnaOptions | None' = None,
    axis: 'Axis | None' = None,
    inplace: 'bool_t' = False,
    limit: 'int | None' = None,
    downcast: 'dict | None | lib.NoDefault' = <no_default>
) -> 'Self | None' method of pandas.core.frame.DataFrame instance
    Fill NA/NaN values using the specified method.

    Parameters
    ----------
    value : scalar, dict, Series, or DataFrame
        Value to use to fill holes (e.g. 0), alternately a
        dict/Series/DataFrame of values specifying which value to use for
        each index (for a Series) or column (for a DataFrame).  Values not
        in the dict/Series/DataFrame will not be filled. This value cannot
        be a list.
    method : {'backfill', 'bfill', 'ffill', None}, default None
        Method to use for filling holes in reindexed Series:

        * ffill: propagate last valid obser

#### Filling a specific column with a value

In [156]:
avg = dfG.Life_Exp.mean()
avg

np.float64(71.77197108045978)

In [157]:
df_temp = dfG.fillna(value={'Life_Exp':avg})
df_temp.to_excel('Datasets/gdp_lexp_na_filled.xlsx')
df_temp

,Country,Life_Exp,GDP,Population,Inc_Class
0,ABW,75.405000,2.962907e+09,107906,H
1,AFE,61.713031,9.099053e+11,607123269,NaN
2,AFG,62.270000,1.913422e+10,33831764,L
3,AFW,56.033996,7.780375e+11,417536152,NaN
4,AGO,61.042000,1.025431e+11,28157798,LM
...,...,...,...,...,...
256,XKX,76.024000,6.295845e+09,1788274,UM
257,YEM,66.761000,4.244449e+10,31159379,L
258,ZAF,64.053000,3.467098e+11,56723537,UM
259,ZMB,60.728000,2.125122e+10,16399089,LM


#### Forward filling

Forward filling replaces a missing value with the previous valid value.

In [161]:
dfE = pd.DataFrame({
    'Date': pd.date_range(start='2026-01-01', periods=10, freq='D'),
    'Electricity_Price': [
        10.5, np.nan, np.nan, 12.0, np.nan,
        13.5, np.nan, 14.0, np.nan, 15.5
    ]
})

dfE

,Date,Electricity_Price
0,2026-01-01,10.5
1,2026-01-02,NaN
2,2026-01-03,NaN
3,2026-01-04,12.0
4,2026-01-05,NaN
5,2026-01-06,13.5
6,2026-01-07,NaN
7,2026-01-08,14.0
8,2026-01-09,NaN
9,2026-01-10,15.5


In [166]:
df_forward = dfE

df_forward['Electricity_Price'] = (
    df_forward['Electricity_Price'].ffill()
)

df_forward

,Date,Electricity_Price
0,2026-01-01,10.5
1,2026-01-02,10.5
2,2026-01-03,10.5
3,2026-01-04,12.0
4,2026-01-05,12.0
5,2026-01-06,13.5
6,2026-01-07,13.5
7,2026-01-08,14.0
8,2026-01-09,14.0
9,2026-01-10,15.5


In [ ]:
dfE = pd.read_csv(r'datasets\elecprice_missingdata.csv')
dfE

#### Backward Fill

Backward filling replaces a missing value with the next valid value.

In [168]:
df_backward = dfE

df_backward['Electricity_Price'] = (
    df_backward['Electricity_Price'].bfill()
)

df_backward

,Date,Electricity_Price
0,2026-01-01,10.5
1,2026-01-02,10.5
2,2026-01-03,10.5
3,2026-01-04,12.0
4,2026-01-05,12.0
5,2026-01-06,13.5
6,2026-01-07,13.5
7,2026-01-08,14.0
8,2026-01-09,14.0
9,2026-01-10,15.5


In [170]:
dfE.to_csv('Datasets/electricity_missingdata.csv',index=False)

Forward filling uses the preceding observation, while backward filling uses the succeeding observation. Both methods preserve existing values, but they may be inappropriate if the carried values do not reasonably represent the missing observations.

[TABLE OF CONTENTS](#contents)

<a id='lesson8'></a>

# 8. Operations

In [173]:
dfD = pd.DataFrame({
    '  Country ': ['Philippines', 'Japan', 'South Korea', 'Thailand'],
    ' GDP ': [471.5, 4213.0, 1871.0, 526.5],
    ' Population  ': [117337000, 123104000, 51712000, 71668000],
    ' Life Expectancy ': [70.5, 84.7, 83.5, 79.7]
})

dfD


,Country,GDP,Population,Life Expectancy
0,Philippines,471.5,117337000,70.5
1,Japan,4213.0,123104000,84.7
2,South Korea,1871.0,51712000,83.5
3,Thailand,526.5,71668000,79.7


In [193]:
dfD.columns

Index(['  Country ', ' GDP ', ' Population  ', ' Life Expectancy '], dtype='object')

Notice that some column labels contain leading or trailing whitespaces.

Use `.str.strip()` to remove whitespaces from the beginning and end of each column label and replace blank spaces between words with underscores using `.str.replace()`:

In [204]:
df_strip = dfD.copy()

df_strip.columns = df_strip.columns.str.strip()

df_strip.columns = df_strip.columns.str.replace(' ', '_')

df_strip

,Country,GDP,Population,Life_Expectancy
0,Philippines,471.5,117337000,70.5
1,Japan,4213.0,123104000,84.7
2,South Korea,1871.0,51712000,83.5
3,Thailand,526.5,71668000,79.7


## 8.1 Statistical operations

Operations in general exclude missing data. `[TMP]`

Performing a descriptive statistic on `axis=rows` (the default)

In [205]:
df_strip['Life_Expectancy'].mean()

np.float64(79.6)

In [212]:
df_strip[['GDP', 'Population', 'Life_Expectancy']].mean()

,0
GDP,1770.5
Population,90955250.0
Life_Expectancy,79.6


Let's try another function

## 8.2 DataFrame's `apply()` method

`DataFrame.apply()` applies a function along input axis of DataFrame.

In [208]:
help(dfD.apply)

Help on method apply in module pandas.core.frame:

apply(
    func: 'AggFuncType',
    axis: 'Axis' = 0,
    raw: 'bool' = False,
    result_type: "Literal['expand', 'reduce', 'broadcast'] | None" = None,
    args=(),
    by_row: "Literal[False, 'compat']" = 'compat',
    engine: "Literal['python', 'numba']" = 'python',
    engine_kwargs: 'dict[str, bool] | None' = None,
    **kwargs
) method of pandas.core.frame.DataFrame instance
    Apply a function along an axis of the DataFrame.

    Objects passed to the function are Series objects whose index is
    either the DataFrame's index (``axis=0``) or the DataFrame's columns
    (``axis=1``). By default (``result_type=None``), the final return type
    is inferred from the return type of the applied function. Otherwise,
    it depends on the `result_type` argument.

    Parameters
    ----------
    func : function
        Function to apply to each column or row.
    axis : {0 or 'index', 1 or 'columns'}, default 0
        Axis along wh

### 8.2.1 DataFrame.apply with `axis=rows`

The default index of `DataFrame.apply` (as with most/all DataFrame methods) is `rows`. Thus, using `apply` without parameters will operate down rows, or inter-rows (effectively, you're geting outputs for the columns)

In [213]:
df_strip.apply(np.sum)

,0
Country,PhilippinesJapanSouth KoreaThailand
GDP,7082.0
Population,363821000
Life_Expectancy,318.4


Of course, we can explictly specify the axis (All these 3 parameter settings will yield the same result: `axis=rows` or `axis=0` or `axis=index`).

### 8.2.2 Using apply with a custom function

`DataFrame.apply` becomes interesting when used with a custom function

Say we want to get the range (highest minus lowest values) of our columns. We could do this with the following:

[TABLE OF CONTENTS](#contents)

# Final Words

Congratulations for reaching this far.

For advanced lessons on `pandas`, the following book is highly suggested: *Pandas for Data Analysis, 2nd Edition *by Wes McKinney

<a id='lesson10'></a>